<a href="https://colab.research.google.com/github/kligeys/AI-Projects/blob/main/multiagent_parallel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install langgraph langchain-openai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 19.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.3
    Uninstalling langchain-core-1.6.3:
      Successfully uninstalled langchain-core-1.6.3


In [48]:

import os
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from openai import OpenAI


In [49]:
# 1. Initialize API Keys
# It's best practice to use environment variables, but you can hardcode them for testing.
# 1. Retrieve the API key safely from Colab Secrets
api_key = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = api_key

In [50]:
# ============================================================
# OPENAI CLIENT
# ============================================================

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)


In [51]:
# ============================================================
# LANGGRAPH STATE
# ============================================================

class AgentState(TypedDict):
    city: str
    stock_symbol: str

    weather_result: str
    stock_result: str

    final_answer: str

In [52]:
def weather_agent(state: AgentState):

    city = state["city"]

    print(f"[Weather Agent] Searching weather for {city}")

    response = client.responses.create(

        model="gpt-5.6",

        tools=[
            {
                "type": "web_search"
            }
        ],

        instructions="""
        You are a Weather Agent.

        Find the current weather for the requested city.

        Use web search to obtain current information.
        Do not guess or invent weather information.

        Return:
        - Temperature
        - Weather condition
        - Any important current weather information
        """,

        input=f"""
        What is the current weather in {city}?
        """
    )

    result = response.output_text

    print("[Weather Agent] Completed")

    return {
        "weather_result": result
    }


In [53]:
# ============================================================
# STOCK AGENT
# ============================================================

def stock_agent(state: AgentState):

    symbol = state["stock_symbol"]

    print(f"[Stock Agent] Searching stock price for {symbol}")

    response = client.responses.create(

        model="gpt-5.6",

        tools=[
            {
                "type": "web_search"
            }
        ],

        instructions="""
        You are a Stock Agent.

        Find the latest available stock price for the
        requested stock symbol.

        Use web search to obtain current information.

        Return:
        - Stock symbol
        - Company name
        - Latest available price
        - Trading date/time if available

        Do not provide investment advice.
        Do not guess the price.
        """,

        input=f"""
        What is the latest stock price for {symbol}?
        """
    )

    result = response.output_text

    print("[Stock Agent] Completed")

    return {
        "stock_result": result
    }


In [54]:
# ============================================================
# SUMMARY AGENT
# ============================================================

def summary_agent(state: AgentState):

    weather = state["weather_result"]
    stock = state["stock_result"]

    print("[Summary Agent] Creating final response")

    prompt = f"""
    Create a concise summary using the information below.

    WEATHER:
    {weather}

    STOCK:
    {stock}

    Present the information clearly.

    Do not provide investment advice.
    """

    response = client.responses.create(

        model="gpt-5.6",

        input=prompt
    )

    return {
        "final_answer": response.output_text
    }


In [55]:
# ============================================================
# BUILD LANGGRAPH
# ============================================================

graph = StateGraph(AgentState)


graph.add_node(
    "weather_agent",
    weather_agent
)

graph.add_node(
    "stock_agent",
    stock_agent
)

graph.add_node(
    "summary_agent",
    summary_agent
)

In [56]:
# ============================================================
# PARALLEL EXECUTION
# ============================================================

graph.add_edge(
    START,
    "weather_agent"
)

graph.add_edge(
    START,
    "stock_agent"
)


# Both agents feed the summary agent

graph.add_edge(
    "weather_agent",
    "summary_agent"
)

graph.add_edge(
    "stock_agent",
    "summary_agent"
)


graph.add_edge(
    "summary_agent",
    END
)


In [57]:
# Compile graph

app = graph.compile()

In [58]:
# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    city = input(
        "Enter city: "
    )

    stock_symbol = input(
        "Enter stock symbol: "
    )

    result = app.invoke(
        {
            "city": city,
            "stock_symbol": stock_symbol,

            "weather_result": "",
            "stock_result": "",

            "final_answer": ""
        }
    )

    print("\n")
    print("=" * 60)
    print("FINAL ANSWER")
    print("=" * 60)

    print(result["final_answer"])

Enter city: trichy
Enter stock symbol: goog
[Stock Agent] Searching stock price for goog
[Weather Agent] Searching weather for trichy
[Weather Agent] Completed
[Stock Agent] Completed
[Summary Agent] Creating final response


FINAL ANSWER
### Trichy Weather
- **38°C (100°F)** and partly sunny
- Thunderstorms possible later this afternoon and evening

### GOOG Stock
- **Alphabet Inc. (Class C): $339.16 USD**
- Latest trade: **September 29, 2026, at 12:15 a.m. UTC**
